### Import

In [1]:
import os
import sys
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow.parquet import ParquetFile
from matplotlib import pyplot as plt 

### General parameters

In [2]:
path_out = "./Data/"
path_timeseries = "Extraction/eICU/Data/Output/"

### Reading Demographic Data

In [3]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'admission.csv')
    all_filenames.append(df_file)
    
df_admission = pd.concat([pd.read_csv(file, low_memory=False) for file in all_filenames if os.path.exists(file)])
df_admission = df_admission.reset_index(drop=True)

In [4]:
df_admission.head(3)

### Reading Missing Percentage

In [3]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'missing_percentage.parquet')
    all_filenames.append(df_file)
    
df_missing = pd.concat([pd.read_parquet(file, engine='pyarrow') for file in all_filenames if os.path.exists(file)])

In [4]:
df_missing.head(3)

### Reading Hourly Averaged Data

In [5]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in (all_stays):
    df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_timeseries.parquet')
    all_filenames.append(df_file)

In [6]:
first_half  = all_filenames[:100000]
second_half = all_filenames[100000:]

In [5]:
first_half_all_data = []

for file in first_half:
    if os.path.exists(file):
        pf = ParquetFile(file) 
        first_rows = next(pf.iter_batches(batch_size = 25)) 
        df = pa.Table.from_batches([first_rows]).to_pandas()
        first_half_all_data.append(df)
        
df_first_half_all_data = pd.concat(first_half_all_data)

In [13]:
df_first_half_all_data = df_first_half_all_data.drop(columns='O2 Admin Device')
df_first_half_all_data.head(3)

In [7]:
second_half_all_data = []

for file in second_half:
    if os.path.exists(file):
        pf = ParquetFile(file) 
        first_rows = next(pf.iter_batches(batch_size = 25)) 
        df = pa.Table.from_batches([first_rows]).to_pandas()
        second_half_all_data.append(df)
        
df_second_half_all_data = pd.concat(second_half_all_data)

In [8]:
df_second_half_all_data = df_second_half_all_data.drop(columns='O2 Admin Device')
df_second_half_all_data.head(3)

### Reading Raw Vitals

In [3]:
vital_sign_col = ['patientunitstayid', 'Bins',
                  'Heart Rate', 'Heart Rate_ind',  
                  'Temperature (C)', 'Temperature (C)_ind',
                  'SpO2', 'SpO2_ind', 
                  'Non-Invasive BP Mean', 'Non-Invasive BP Mean_ind',
                  'Non-Invasive BP Diastolic', 'Non-Invasive BP Diastolic_ind',
                  'Non-Invasive BP Systolic', 'Non-Invasive BP Systolic_ind',
                  'Respiratory Rate', 'Respiratory Rate_ind',
                  'ST1', 'ST1_ind', 
                  'ST2', 'ST2_ind', 
                  'ST3', 'ST3_ind']

In [4]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in (all_stays):
    df_file = os.path.join(path_timeseries, str(stay_id), 'imputed_vital_timeseries.parquet')
    all_filenames.append(df_file)

In [5]:
all_vitals_data = []

for file in all_filenames:
    if os.path.exists(file):
        pf = ParquetFile(file) 
        first_ten_rows = next(pf.iter_batches(batch_size = 290, columns=vital_sign_col)) 
        df = pa.Table.from_batches([first_ten_rows]).to_pandas()
        all_vitals_data.append(df)
        
df_raw_vitals = pd.concat(all_vitals_data)

In [6]:
df_raw_vitals.head(3)

### Reading Processed Vital Signs

In [3]:
all_stays  = pd.Series(os.listdir(path_timeseries))
all_filenames = []

for stay_id in all_stays:
    df_file = os.path.join(path_timeseries, str(stay_id), 'vitals_processed.parquet')
    all_filenames.append(df_file)

In [4]:
first_half  = all_filenames[:100000]
second_half = all_filenames[100000:]

In [5]:
first_half_vitals_data = []

for file in first_half:
    if os.path.exists(file):
        pf = ParquetFile(file) 
        first_ten_rows = next(pf.iter_batches(batch_size = 22)) 
        df = pa.Table.from_batches([first_ten_rows]).to_pandas()
        first_half_vitals_data.append(df)
        
df_processed_vitals_first_half = pd.concat(first_half_vitals_data)

In [6]:
df_processed_vitals_first_half.head(3)

In [5]:
second_half_vitals_data = []

for file in second_half:
    if os.path.exists(file):
        pf = ParquetFile(file) 
        first_ten_rows = next(pf.iter_batches(batch_size = 22)) 
        df = pa.Table.from_batches([first_ten_rows]).to_pandas()
        second_half_vitals_data.append(df)
        
df_processed_vitals_second_half = pd.concat(second_half_vitals_data)

In [6]:
df_processed_vitals_second_half.head(3)

### Missing in 24H

In [5]:
missing_value_df = df.groupby('patientid').apply(lambda x: x.isnull().all())
missing_value_df.drop(columns=['patientid'], inplace=True)
missing_value_df = missing_value_df.reset_index()
missing_value_df = missing_value_df.replace(True, np.nan)
percent_missing = missing_value_df.isnull().sum() * 100 / len(missing_value_df)
missing_value_perc = pd.DataFrame({'column_name': missing_value_df.columns, 'percent_missing': percent_missing})
missing_value_perc.sort_values('percent_missing', inplace=True, ascending=False)
missing_value_perc.reset_index(inplace=True, drop=True)
missing_value_perc.head()

,column_name,percent_missing
0,vasopressors,100.000000
1,Chemotherapie,99.988170
2,Antihelmenticum,99.958593
3,Others in Case of HIT,99.931975
4,Mineralokortikoid,99.923102


### Save Data

In [5]:
df_admission.to_csv(path_out + 'demographic_all.csv', index=False)

In [5]:
df_missing_file_path = os.path.join(path_out, 'missing_all.parquet')
df_missing_table = pa.Table.from_pandas(df_missing)
pq.write_table(df_missing_table, df_missing_file_path)

In [14]:
df_all_file_first_half_path = os.path.join(path_out, 'all_24h_data_first_half.parquet')
df_first_half_all_table = pa.Table.from_pandas(df_first_half_all_data)
pq.write_table(df_first_half_all_table, df_all_file_first_half_path)

In [9]:
df_all_file_second_half_path = os.path.join(path_out, 'all_24h_data_second_half.parquet')
df_second_half_all_table = pa.Table.from_pandas(df_second_half_all_data)
pq.write_table(df_second_half_all_table, df_all_file_second_half_path)

In [7]:
df_raw_vitals_file_path = os.path.join(path_out, 'vitals_5min_24h_data.parquet')
df_raw_vitals_table = pa.Table.from_pandas(df_raw_vitals)
pq.write_table(df_raw_vitals_table, df_raw_vitals_file_path)

In [7]:
df_processed_vitals_first_half_file_path = os.path.join(path_out, 'vitals_processed_data_first_half.parquet')
df_processed_vitals_first_half_table = pa.Table.from_pandas(df_processed_vitals_first_half)
pq.write_table(df_processed_vitals_first_half_table, df_processed_vitals_first_half_file_path)

In [7]:
df_processed_vitals_second_half_file_path = os.path.join(path_out, 'vitals_processed_data_second_half.parquet')
df_processed_vitals_second_half_table = pa.Table.from_pandas(df_processed_vitals_second_half)
pq.write_table(df_processed_vitals_second_half_table, df_processed_vitals_second_half_file_path)

### Columns

In [ ]:
all_variables = [
    
'pastHistory', 'ICD-9', 'ICD-10', 'ICD-10_Embedding', 
'elixhauser_comorbidity', 'elixhauser_readmission', 'elixhauser_mortalityrisks',
 
'Fentanyl_PRC', 'Propofol_PRC', 'Norepinephrine_PRC', 'Insulin_PRC', 'Midazolam_PRC', 'Heparin_PRC', 
'Dexmedetomidine_PRC', 'Amiodarone_PRC', 'Vasopressin_PRC', 'Phenylephrine_PRC', 'Dopamine_PRC', 'Nicardipine_PRC',
'Milrinone_PRC', 'Pantoprazole_PRC', 'Diltiazem_PRC', 'Dobutamine_PRC', 'Nitroglycerin_PRC', 'Epinephrine_PRC',
'Antibiotic_PRC', 'Warfarin_PRC', 'Vasopressors',
    
'Urine_IO', 'Propofol_IO', 'Fentanyl_IO', 'Insulin_IO', 'Heparin_IO', 'Midazolam_IO', 'Dexmedetomidine_IO', 
'Vassopressin_IO', 'Albumin_IO', 'Ceftriaxone_IO', 'Cefazolin_IO', 'Cefepime_IO', 'Ceftazidime_IO', 
'Vancomycin_IO', 'Clindamycin_IO', 'Metronidazole_IO', 'Meropenem_IO', 'Acyclovir_IO', 'Azithromycin_IO', 
'Levofloxacin_IO', 'Micafungin_IO', 'Fluconazole_IO', 'Thiamine_IO', 'Dobutamine_IO', 'Milrinone_IO', 
'Fluids_IO', 'OralIntake_IO', 'P.O._IO', 'SodiumChloride_IO', 'IVPB_IO', 'Stool_IO', 'Crystalloids_IO',
'NSIVF_IO', 'Norepinephrine_IO', 'Amiodarone_IO', 'Phenylephrine_IO', 'Epinephrine_IO', 'Nicardipine_IO',
'Pantoprazole_IO', 'Diltiazem_IO', 'Nitroglycerin_IO',
 
'Non-Invasive BP Mean', 'Non-Invasive BP Systolic', 'Non-Invasive BP Diastolic', 
'Invasive BP Systolic', 'Invasive BP Diastolic', 'Invasive BP Mean', 'PA Systolic', 'PA Diastolic', 'PA Mean', 
'Temperature (C)', 'Heart Rate', 'Respiratory Rate', 'CVP', 'ETCO2', 'ST1', 'ST2', 'ST3',
 
'PT', 'PTT', 'PT - INR', 'pH', 'lactate', 'LDH',  'Base Excess', 'anion gap', 'Bicarbonate', 'creatinine',     
'Hct', 'Hgb', 'total bilirubin', 'direct bilirubin', 'MPV', 'MCV', 'MCH', 'MCHC',   
'RDW', 'RBC', 'WBC x 1000', 'platelets x 1000', 'Glucose', 'ammonia', 'magnesium', 'phosphate', 'alkaline phos.',    
'potassium',  'sodium', 'chloride', 'calcium', 'ionized calcium', 'total cholesterol', 'C-Reactive Protein',
'paO2', 'paCO2', 'ALT (SGPT)', 'AST (SGOT)', '-bands', '-polys', 'amylase', 'lipase', '-lymphs', '-monos',    
'-eos', '-basos', 'LPM O2', 'O2 Content', 'O2 Saturation', 'Total CO2', 'albumin', 'troponin - T', 'troponin - I',    
'Vancomycin - peak', 'Vancomycin - trough', 'Vancomycin - random', 'triglycerides', 'fibrinogen', 'transferrin',
'Ferritin', 'cortisol', 'total protein', 'PEEP', 'Tidal Volume', 'Vent Rate', 'FiO2', 'BUN', 'TSH',
'Pressure Support', 'Pressure Control', 'Peak Airway/Pressure', 'Pain Goal', 'Pain Score', 'Pain Present',  
'GCS Total', 'Motor', 'Verbal', 'Eyes', 'RASS', 'Fall Risk',  'Delirium Score', 'Delirium Scale',
'Symptoms of Delirium Present', 'Sedation Goal', 'Sedation Score', 'Flow Rate', 'SpO2', 'SVO2',
'Pulse', 'O2 Admin Device', 'MAP (mmHg)', 'Total Respiratory Rate', 'Exhaled MV', 'Exhaled Vt', 
'Exhaled TV (patient)', 'Exhaled TV (machine)', 'Peak Pressure', 'Plateau Pressure', 'Peak Insp. Pressure', 
'Mean Airway Pressure', 'Inspiratory Flow Rate', 'O2 Percentage', 'Oxygen Flow Rate',
'Ventilator Type', 
    
'FiO2 (Set)', 'Pressure Support (Set)', 'PEEP (Set)', 'LPM O2 (Set)', 'Vent Rate (Set)', 'Tidal Volume (Set)',
'TV/kg IBW (Set)', 'PEEP/CPAP (Set)', 'Flow Sensitivity (Set)', 'Peak Flow (Set)', 'Bodyweight']

cat_int_value = ['Pain Goal', 'Pain Score', 'GCS Total', 'Motor', 'Verbal', 'Eyes', 'RASS', 'Fall Risk',
                 'Delirium Score', 'Symptoms of Delirium Present', 'Sedation Goal', 'Sedation Score', 
                 'Ventilator Type', 'Pain Present']

cat_text_value = ['Delirium Scale', 'O2 Admin Device']